In [1]:
import os
os.chdir("..")
os.makedirs("app", exist_ok=True)
print(os.getcwd())

/home/s2025ae05941/workspace/heart-mlops


In [2]:
%%writefile app/main.py
"""FastAPI service that serves the heart-disease model.

Endpoints:
  GET  /         - basic info
  GET  /health   - is the API up and the model loaded?
  POST /predict  - predict heart-disease risk for one patient
  GET  /metrics  - Prometheus metrics (used for monitoring in Task 8)
"""
import logging
import time
from typing import Optional

from fastapi import FastAPI, Request
from fastapi.responses import Response
from prometheus_client import CONTENT_TYPE_LATEST, Counter, Histogram, generate_latest
from pydantic import BaseModel, ConfigDict

from src.predict import SAMPLE_PATIENT, load_model, predict

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("heart-api")

app = FastAPI(title="Heart Disease Risk API", version="1.0")
model = load_model()

REQUESTS = Counter("api_requests_total", "Total API requests", ["endpoint", "status"])
LATENCY = Histogram("api_request_latency_seconds", "Request latency in seconds", ["endpoint"])
PREDICTIONS = Counter("predictions_total", "Predictions made, by label", ["label"])


class Patient(BaseModel):
    """The 13 clinical features of one patient."""
    model_config = ConfigDict(json_schema_extra={"example": SAMPLE_PATIENT})

    age: float
    sex: float
    cp: float
    trestbps: float
    chol: float
    fbs: float
    restecg: float
    thalach: float
    exang: float
    oldpeak: float
    slope: float
    ca: Optional[float] = None
    thal: Optional[float] = None


@app.middleware("http")
async def log_and_measure(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    elapsed = time.perf_counter() - start
    path = request.url.path
    if path != "/metrics":
        REQUESTS.labels(path, str(response.status_code)).inc()
        LATENCY.labels(path).observe(elapsed)
    log.info("%s %s -> %s (%.1f ms)", request.method, path,
             response.status_code, elapsed * 1000)
    return response


@app.get("/")
def root():
    return {"message": "Heart Disease Risk API. See /docs for usage."}


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}


@app.post("/predict")
def predict_endpoint(patient: Patient):
    result = predict([patient.model_dump()], model=model)[0]
    PREDICTIONS.labels(result["label"]).inc()
    log.info("prediction=%s confidence=%s", result["label"], result["confidence"])
    return result


@app.get("/metrics")
def metrics():
    return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

Writing app/main.py


In [3]:
%%writefile requirements-api.txt
numpy==2.0.2
pandas==2.3.3
scikit-learn==1.6.1
scipy==1.13.1
joblib==1.5.3
fastapi==0.115.12
uvicorn==0.34.3
prometheus-client==0.22.1

Writing requirements-api.txt


In [4]:
%%writefile Dockerfile
# Small official Python image (from the AWS mirror to avoid Docker Hub rate limits)
FROM public.ecr.aws/docker/library/python:3.9-slim

WORKDIR /app

# Install libraries first so Docker can cache this layer
COPY requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt

# Copy the code and the trained model
COPY src/ src/
COPY app/ app/
COPY models/ models/

EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [5]:
%%writefile .dockerignore
.venv
.git
mlruns
data
notebooks
reports
tests

Writing .dockerignore


In [7]:
import os
os.makedirs("k8s", exist_ok=True)
print(os.getcwd())

/home/s2025ae05941/workspace/heart-mlops


In [8]:
%%writefile k8s/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: heart-api
  labels:
    app: heart-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: heart-api
  template:
    metadata:
      labels:
        app: heart-api
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "8000"
        prometheus.io/path: "/metrics"
    spec:
      containers:
        - name: heart-api
          image: heart-api:1.0
          imagePullPolicy: Never
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: 100m
              memory: 256Mi
            limits:
              cpu: 500m
              memory: 512Mi
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 10
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 20

Writing k8s/deployment.yaml


In [9]:
%%writefile k8s/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: heart-api
  labels:
    app: heart-api
spec:
  type: LoadBalancer
  selector:
    app: heart-api
  ports:
    - name: http
      port: 80
      targetPort: 8000
      nodePort: 30080

Writing k8s/service.yaml
